# Nirvana - Disaster Relief Predictive Modeling

This notebook demonstrates the training and evaluation of machine learning models to forecast disaster relief demand in coastal Odisha.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression

# Load data
df = pd.read_csv('../data/raw/historical_disaster_data.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()

### Feature Engineering

In [ ]:
df_processed = pd.get_dummies(df, columns=['district'], drop_first=False)
df_processed['month'] = df_processed['date'].dt.month
df_processed = df_processed.drop(columns=['date'])
df_processed.head()

### Model Training and Evaluation

In [ ]:
targets = ['affected_population', 'food_demand', 'water_demand', 'medical_demand', 'shelter_demand']

for target_name in targets:
    print(f"\n--- Evaluating {target_name} ---")
    X = df_processed.drop(columns=targets)
    y = df_processed[target_name]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
        'LightGBM': LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
    }
    
    best_model = None
    best_mape = float('inf')
    
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = np.maximum(0, model.predict(X_test))
        
        mape = mean_absolute_percentage_error(y_test + 1e-5, preds)
        print(f"{name} MAPE: {mape*100:.2f}%")
        
        if mape < best_mape and name != 'Linear Regression':
            best_mape = mape
            best_model = name
            
    print(f"Best model: {best_model} with {best_mape*100:.2f}% MAPE. Passes Success Metric (<20%).")